# The EM Algorithm

In [ ]:
%pip install numpy matplotlib scipy scikit-learn

## Motivation: Mixture Models and Incomplete Data

In the optimization lectures, we studied methods for maximizing log-likelihood functions: Newton-Raphson, Fisher Scoring, BFGS, and gradient descent. These methods work well when the log-likelihood has a tractable form. But what happens when part of the data is missing or unobserved?

Consider a concrete example. Suppose we measure the heights of 300 individuals from a population that is actually a *mixture* of two subpopulations (say, two different age groups), but we do not know which subpopulation each individual belongs to. We model the observed heights as coming from a two-component Gaussian mixture:

$$f(y_i | \boldsymbol{\theta}) = \pi \, \phi(y_i | \mu_1, \sigma_1^2) + (1 - \pi) \, \phi(y_i | \mu_2, \sigma_2^2)$$

where $\phi(\cdot | \mu, \sigma^2)$ denotes the normal density, $\pi$ is the mixing proportion, and $\boldsymbol{\theta} = (\pi, \mu_1, \sigma_1^2, \mu_2, \sigma_2^2)$.

The observed-data log-likelihood is:

$$\ell(\boldsymbol{\theta}) = \sum_{i=1}^{n} \log \left[ \pi \, \phi(y_i | \mu_1, \sigma_1^2) + (1 - \pi) \, \phi(y_i | \mu_2, \sigma_2^2) \right]$$

The log of a sum has no closed-form MLE. Unlike the single-normal case where the log-likelihood decomposes into a simple sum of terms, here each term involves a sum inside the logarithm. Taking derivatives and setting them to zero yields a system of equations with no analytical solution.

Let us simulate this scenario and visualize the data:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

np.random.seed(42)

# True parameters
n = 300
pi_true = 0.4
mu1_true, sigma1_true = 62, 3.0
mu2_true, sigma2_true = 70, 2.5

# Simulate mixture data
z_true = np.random.binomial(1, 1 - pi_true, n)  # 0 = component 1, 1 = component 2
y = np.where(
    z_true == 0,
    np.random.normal(mu1_true, sigma1_true, n),
    np.random.normal(mu2_true, sigma2_true, n),
)

# Plot histogram with true density overlay
x_grid = np.linspace(50, 80, 500)
true_density = pi_true * stats.norm.pdf(x_grid, mu1_true, sigma1_true) + (
    1 - pi_true
) * stats.norm.pdf(x_grid, mu2_true, sigma2_true)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y, bins=35, density=True, alpha=0.6, color="steelblue", edgecolor="white")
ax.plot(x_grid, true_density, "k-", linewidth=2, label="True mixture density")
ax.plot(
    x_grid,
    pi_true * stats.norm.pdf(x_grid, mu1_true, sigma1_true),
    "--",
    color="tab:orange",
    label=f"Component 1 ($\\mu_1={mu1_true}$)",
)
ax.plot(
    x_grid,
    (1 - pi_true) * stats.norm.pdf(x_grid, mu2_true, sigma2_true),
    "--",
    color="tab:green",
    label=f"Component 2 ($\\mu_2={mu2_true}$)",
)
ax.set_xlabel("Height (inches)")
ax.set_ylabel("Density")
ax.set_title("Observed Heights from a Two-Component Gaussian Mixture")
ax.legend()
plt.tight_layout()


### Complete vs. Incomplete Data

The key insight is that if we *knew* which component each observation came from, estimation would be straightforward. Let $z_i \in \{1, 2\}$ denote the (unobserved) component membership for observation $i$. The pair $(y_i, z_i)$ is called the **complete data**, while the observed $y_i$ alone is the **incomplete data**. The variable $z_i$ is a **latent variable**.

The **complete-data log-likelihood** is:

$$\ell_c(\boldsymbol{\theta}) = \sum_{i=1}^{n} \left[ \mathbb{1}(z_i = 1) \left( \log \pi + \log \phi(y_i | \mu_1, \sigma_1^2) \right) + \mathbb{1}(z_i = 2) \left( \log(1 - \pi) + \log \phi(y_i | \mu_2, \sigma_2^2) \right) \right]$$

This is a sum of logs (not the log of a sum), and each parameter appears in only a subset of terms. Maximization with respect to each parameter has a closed-form solution, given the $z_i$'s.

This is the fundamental structure the EM algorithm exploits: the complete-data problem is easy, and EM iterates between inferring the missing data and maximizing the complete-data likelihood.

**Connection to optimization:** We *could* try to directly optimize the observed-data log-likelihood $\ell(\boldsymbol{\theta})$ using BFGS or Newton-Raphson. However, the mixture log-likelihood surface often has multiple local maxima, and gradient-based methods can be sensitive to initialization. The EM algorithm provides a principled, stable alternative that is naturally suited to problems with latent variables.

## The EM Algorithm Framework

The EM (Expectation-Maximization) algorithm, introduced by Dempster, Laird, and Rubin (1977), is a general iterative method for finding maximum likelihood estimates when the data are incomplete.

### Setup

Let $\mathbf{Y}$ denote the observed (incomplete) data, $\mathbf{Z}$ the missing or latent data, and $\boldsymbol{\theta}$ the parameter vector. The complete data is $(\mathbf{Y}, \mathbf{Z})$. We want to maximize the observed-data log-likelihood $\ell(\boldsymbol{\theta}) = \log f(\mathbf{Y} | \boldsymbol{\theta})$, but it is easier to work with the complete-data log-likelihood $\log f(\mathbf{Y}, \mathbf{Z} | \boldsymbol{\theta})$.

### The Q-function

Define the **Q-function** as the expected complete-data log-likelihood, where the expectation is taken over the conditional distribution of $\mathbf{Z}$ given $\mathbf{Y}$ and the current parameter estimate $\boldsymbol{\theta}^{(t)}$:

$$Q(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)}) = E_{\mathbf{Z} | \mathbf{Y}, \boldsymbol{\theta}^{(t)}} \left[ \log f(\mathbf{Y}, \mathbf{Z} | \boldsymbol{\theta}) \right]$$

The EM algorithm alternates between two steps:

**E-step (Expectation):** Compute $Q(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)})$. In practice, this means computing the conditional expectations of the sufficient statistics of the complete-data distribution, given the observed data and current parameter estimates.

**M-step (Maximization):** Find $\boldsymbol{\theta}^{(t+1)} = \arg\max_{\boldsymbol{\theta}} Q(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)})$. Because $Q$ typically has the tractable form of the complete-data log-likelihood (with latent variables replaced by their conditional expectations), this maximization is often available in closed form.

### Pseudocode

```
EM Algorithm:
1. Initialize parameters θ^(0)
2. For t = 0, 1, 2, ... until convergence:
   a. E-step: Compute Q(θ | θ^(t)) = E[log f(Y, Z | θ) | Y, θ^(t)]
   b. M-step: Set θ^(t+1) = argmax_θ Q(θ | θ^(t))
3. Return θ^(t+1)
```

### Why Does EM Work?

The key property is that **each EM iteration is guaranteed to increase (or at least not decrease) the observed-data log-likelihood**:

$$\ell(\boldsymbol{\theta}^{(t+1)}) \geq \ell(\boldsymbol{\theta}^{(t)})$$

The intuition comes from Jensen's inequality. We can decompose the observed-data log-likelihood as:

$$\ell(\boldsymbol{\theta}) = Q(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)}) - H(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)})$$

where $H(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)}) = E_{\mathbf{Z} | \mathbf{Y}, \boldsymbol{\theta}^{(t)}} \left[ \log f(\mathbf{Z} | \mathbf{Y}, \boldsymbol{\theta}) \right]$. One can show (using Jensen's inequality applied to the log function) that $H(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)}) \leq H(\boldsymbol{\theta}^{(t)} | \boldsymbol{\theta}^{(t)})$. Since the M-step ensures $Q(\boldsymbol{\theta}^{(t+1)} | \boldsymbol{\theta}^{(t)}) \geq Q(\boldsymbol{\theta}^{(t)} | \boldsymbol{\theta}^{(t)})$, combining these inequalities gives the monotone increase property.

### Connection to Optimization Methods

EM shares the iterative structure of Newton-Raphson and BFGS, but differs in important ways:

| Property | Newton-Raphson / BFGS | EM |
|---|---|---|
| Requires gradient | Yes | No (typically) |
| Requires Hessian | Yes (NR) / approximated (BFGS) | No |
| Monotone likelihood increase | Not guaranteed | Yes |
| Convergence rate | Quadratic (NR) / superlinear (BFGS) | Linear |
| Handles constraints naturally | No | Often yes (e.g., $\pi \in [0,1]$) |
| Parameter updates interpretable | Rarely | Often (weighted MLEs) |

EM tends to converge more slowly (linearly rather than quadratically), but its stability and simplicity often make it the preferred choice for latent-variable models. The convergence rate here refers to the rate at which the parameter estimates $\boldsymbol{\theta}^{(t)}$ approach the MLE $\hat{\boldsymbol{\theta}}$. Linear convergence means the error $\|\boldsymbol{\theta}^{(t+1)} - \hat{\boldsymbol{\theta}}\| \approx r \|\boldsymbol{\theta}^{(t)} - \hat{\boldsymbol{\theta}}\|$ shrinks by a constant factor $r \in (0, 1)$ per iteration. Quadratic convergence (Newton-Raphson) means the error is roughly squared each step, so it vanishes much faster near the solution.

### Demonstrating the Challenge of Direct Optimization

Let us see what happens when we try to optimize the mixture log-likelihood directly using BFGS with different starting points:

In [ ]:
def neg_loglik_mixture(params, y):
    """Negative log-likelihood for 2-component Gaussian mixture."""
    pi, mu1, log_s1, mu2, log_s2 = params
    sigma1, sigma2 = np.exp(log_s1), np.exp(log_s2)
    # Clamp pi to (0, 1) for numerical stability
    pi = np.clip(pi, 1e-6, 1 - 1e-6)
    ll = np.sum(
        np.log(
            pi * stats.norm.pdf(y, mu1, sigma1)
            + (1 - pi) * stats.norm.pdf(y, mu2, sigma2)
        )
    )
    return -ll


# Try BFGS from 10 random initializations
np.random.seed(123)
results = []
for i in range(10):
    init = [
        np.random.uniform(0.2, 0.8),
        np.random.uniform(55, 75),
        np.log(np.random.uniform(1, 5)),
        np.random.uniform(55, 75),
        np.log(np.random.uniform(1, 5)),
    ]
    res = minimize(neg_loglik_mixture, init, args=(y,), method="L-BFGS-B")
    results.append((-res.fun, res.x))

# Display results
print(f"{'Init':>4s}  {'LogLik':>10s}  {'pi':>6s}  {'mu1':>6s}  {'sig1':>6s}  {'mu2':>6s}  {'sig2':>6s}")
print("-" * 60)
for i, (ll, p) in enumerate(results):
    pi_est = np.clip(p[0], 0, 1)
    print(
        f"{i+1:>4d}  {ll:>10.2f}  {pi_est:>6.3f}  {p[1]:>6.2f}  "
        f"{np.exp(p[2]):>6.2f}  {p[3]:>6.2f}  {np.exp(p[4]):>6.2f}"
    )

Different initializations often converge to different local optima, with some producing degenerate solutions. The EM algorithm, while not immune to local optima, tends to be more stable and always produces valid parameter estimates that respect the natural constraints of the model.

### Question

The **Generalized EM (GEM)** algorithm replaces the M-step with any update that *increases* $Q(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)})$ rather than fully maximizing it. That is, $Q(\boldsymbol{\theta}^{(t+1)} | \boldsymbol{\theta}^{(t)}) \geq Q(\boldsymbol{\theta}^{(t)} | \boldsymbol{\theta}^{(t)})$, but $\boldsymbol{\theta}^{(t+1)}$ need not be the global maximizer of $Q$.

Does the observed-data log-likelihood still increase at each iteration under GEM? Why might this variant be useful?

### Answer

Yes, the observed-data log-likelihood still increases at each GEM iteration. The proof of monotone increase only requires that $Q(\boldsymbol{\theta}^{(t+1)} | \boldsymbol{\theta}^{(t)}) \geq Q(\boldsymbol{\theta}^{(t)} | \boldsymbol{\theta}^{(t)})$, not that $\boldsymbol{\theta}^{(t+1)}$ maximizes $Q$. The inequality $H(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)}) \leq H(\boldsymbol{\theta}^{(t)} | \boldsymbol{\theta}^{(t)})$ holds for any $\boldsymbol{\theta}$, so any improvement in $Q$ translates to an improvement in $\ell(\boldsymbol{\theta})$.

GEM is useful when the M-step does not have a closed-form solution. Instead of solving a difficult optimization problem exactly, you can take a single Newton-Raphson or gradient ascent step on $Q$, which is often sufficient. This makes each iteration cheaper while preserving the monotone convergence guarantee.

## Gaussian Mixture Models: EM Derivation and Implementation

We now derive the E-step and M-step for the two-component Gaussian mixture in detail, then implement the algorithm from scratch.

### E-step: Computing Responsibilities

Given current parameter estimates $\boldsymbol{\theta}^{(t)} = (\pi^{(t)}, \mu_1^{(t)}, \sigma_1^{2(t)}, \mu_2^{(t)}, \sigma_2^{2(t)})$, the E-step computes the posterior probability that each observation $y_i$ came from component 1. This quantity is called the **responsibility** of component 1 for observation $i$:

$$\gamma_i^{(t)} = P(z_i = 1 | y_i, \boldsymbol{\theta}^{(t)}) = \frac{\pi^{(t)} \, \phi(y_i | \mu_1^{(t)}, \sigma_1^{2(t)})}{\pi^{(t)} \, \phi(y_i | \mu_1^{(t)}, \sigma_1^{2(t)}) + (1 - \pi^{(t)}) \, \phi(y_i | \mu_2^{(t)}, \sigma_2^{2(t)})}$$

This is simply Bayes' rule. The responsibility $\gamma_i^{(t)}$ is a "soft" assignment: it ranges continuously from 0 to 1, reflecting our uncertainty about which component generated $y_i$.

Substituting these responsibilities into the complete-data log-likelihood, the resulting Q-function for the two-component Gaussian mixture is:

$$Q(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)}) = \sum_{i=1}^n \left[ \gamma_i^{(t)} \left( \log \pi + \log \phi(y_i | \mu_1, \sigma_1^2) \right) + (1 - \gamma_i^{(t)}) \left( \log(1 - \pi) + \log \phi(y_i | \mu_2, \sigma_2^2) \right) \right]$$

This has the same form as the complete-data log-likelihood, but with the unknown indicators $\mathbb{1}(z_i = 1)$ replaced by their conditional expectations $\gamma_i^{(t)}$. Because the Q-function is a weighted sum of log-densities from each component, maximizing with respect to $(\pi, \mu_1, \sigma_1^2)$ and $(\mu_2, \sigma_2^2)$ separates into independent weighted MLE problems.

### M-step: Weighted Maximum Likelihood Estimates

Given the responsibilities, the M-step updates each parameter by maximizing the Q-function. This amounts to computing weighted MLEs, where observation $i$ contributes to component 1 with weight $\gamma_i$ and to component 2 with weight $(1 - \gamma_i)$:

$$\pi^{(t+1)} = \frac{1}{n} \sum_{i=1}^n \gamma_i^{(t)}$$

$$\mu_1^{(t+1)} = \frac{\sum_{i=1}^n \gamma_i^{(t)} \, y_i}{\sum_{i=1}^n \gamma_i^{(t)}}, \quad \mu_2^{(t+1)} = \frac{\sum_{i=1}^n (1 - \gamma_i^{(t)}) \, y_i}{\sum_{i=1}^n (1 - \gamma_i^{(t)})}$$

$$\sigma_1^{2(t+1)} = \frac{\sum_{i=1}^n \gamma_i^{(t)} (y_i - \mu_1^{(t+1)})^2}{\sum_{i=1}^n \gamma_i^{(t)}}, \quad \sigma_2^{2(t+1)} = \frac{\sum_{i=1}^n (1 - \gamma_i^{(t)}) (y_i - \mu_2^{(t+1)})^2}{\sum_{i=1}^n (1 - \gamma_i^{(t)})}$$

Each of these has the form of a standard MLE with observations weighted by their responsibilities. This is what makes EM intuitive: the E-step figures out "who belongs where" (softly), and the M-step fits each component to its weighted share of the data.

### Implementation

In [ ]:
def em_gmm(y, pi_init, mu1_init, sigma1_init, mu2_init, sigma2_init,
           max_iter=200, tol=1e-8):
    """EM algorithm for a 2-component Gaussian mixture model.

    Parameters
    ----------
    y : array of shape (n,)
        Observed data.
    pi_init, mu1_init, sigma1_init, mu2_init, sigma2_init : float
        Initial parameter values (sigma = standard deviation).
    max_iter : int
        Maximum number of iterations.
    tol : float
        Convergence tolerance on log-likelihood change.

    Returns
    -------
    dict with keys: pi, mu1, sigma1, mu2, sigma2, gammas, loglik_history
    """
    n = len(y)
    pi = pi_init
    mu1, mu2 = mu1_init, mu2_init
    s1, s2 = sigma1_init, sigma2_init
    loglik_history = []

    for iteration in range(max_iter):
        # E-step: compute responsibilities
        d1 = pi * stats.norm.pdf(y, mu1, s1)
        d2 = (1 - pi) * stats.norm.pdf(y, mu2, s2)
        gamma = d1 / (d1 + d2)

        # Compute observed-data log-likelihood
        ll = np.sum(np.log(d1 + d2))
        loglik_history.append(ll)

        # Check convergence
        if iteration > 0 and abs(loglik_history[-1] - loglik_history[-2]) < tol:
            break

        # M-step: update parameters
        n1 = np.sum(gamma)
        n2 = n - n1

        pi = n1 / n
        mu1 = np.sum(gamma * y) / n1
        mu2 = np.sum((1 - gamma) * y) / n2
        s1 = np.sqrt(np.sum(gamma * (y - mu1) ** 2) / n1)
        s2 = np.sqrt(np.sum((1 - gamma) * (y - mu2) ** 2) / n2)

    return {
        "pi": pi,
        "mu1": mu1,
        "sigma1": s1,
        "mu2": mu2,
        "sigma2": s2,
        "gammas": gamma,
        "loglik_history": loglik_history,
    }

Let us run this on our simulated data:

In [ ]:
result = em_gmm(y, pi_init=0.5, mu1_init=58, sigma1_init=4,
                mu2_init=72, sigma2_init=4)

print("EM estimates vs. true values:")
print(f"  pi:     {result['pi']:.3f}  (true: {pi_true})")
print(f"  mu1:    {result['mu1']:.2f}  (true: {mu1_true})")
print(f"  sigma1: {result['sigma1']:.2f}  (true: {sigma1_true})")
print(f"  mu2:    {result['mu2']:.2f}  (true: {mu2_true})")
print(f"  sigma2: {result['sigma2']:.2f}  (true: {sigma2_true})")
print(f"  Iterations: {len(result['loglik_history'])}")

### Convergence Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Log-likelihood curve
axes[0].plot(result["loglik_history"], "o-", markersize=3, color="steelblue")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Log-likelihood")
axes[0].set_title("EM Convergence: Log-Likelihood")

# Density evolution: run EM step-by-step and plot at selected iterations
show_iters = [0, 1, 3, 10, 50]
pi_t, mu1_t, mu2_t = 0.5, 58.0, 72.0
s1_t, s2_t = 4.0, 4.0
x_grid = np.linspace(50, 80, 500)

for iteration in range(max(show_iters) + 1):
    if iteration in show_iters:
        dens = pi_t * stats.norm.pdf(x_grid, mu1_t, s1_t) + (
            1 - pi_t
        ) * stats.norm.pdf(x_grid, mu2_t, s2_t)
        axes[1].plot(x_grid, dens, label=f"Iter {iteration}", alpha=0.8)

    # E-step
    d1 = pi_t * stats.norm.pdf(y, mu1_t, s1_t)
    d2 = (1 - pi_t) * stats.norm.pdf(y, mu2_t, s2_t)
    gamma = d1 / (d1 + d2)

    # M-step
    n1 = np.sum(gamma)
    n2 = n - n1
    pi_t = n1 / n
    mu1_t = np.sum(gamma * y) / n1
    mu2_t = np.sum((1 - gamma) * y) / n2
    s1_t = np.sqrt(np.sum(gamma * (y - mu1_t) ** 2) / n1)
    s2_t = np.sqrt(np.sum((1 - gamma) * (y - mu2_t) ** 2) / n2)

axes[1].hist(y, bins=35, density=True, alpha=0.3, color="grey")
axes[1].set_xlabel("Height (inches)")
axes[1].set_ylabel("Density")
axes[1].set_title("Estimated Density at Selected Iterations")
axes[1].legend()

plt.tight_layout()


### Soft Clustering via Responsibilities

The responsibilities $\gamma_i$ give a natural soft clustering of the data. Each observation is assigned a probability of belonging to each component rather than a hard label.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
order = np.argsort(y)
colors = plt.cm.RdYlBu(result["gammas"][order])
ax.scatter(y[order], np.zeros_like(y), c=result["gammas"][order],
           cmap="RdYlBu", s=20, alpha=0.8, edgecolors="none")
ax.set_xlabel("Height (inches)")
ax.set_yticks([])
ax.set_title("Observations Colored by Responsibility ($\\gamma_i$: blue = component 1, red = component 2)")
cbar = plt.colorbar(ax.collections[0], ax=ax, orientation="horizontal",
                     pad=0.35, aspect=40)
cbar.set_label("$\\gamma_i$ (responsibility for component 1)")
plt.tight_layout()


### Comparison with scikit-learn

The `sklearn.mixture.GaussianMixture` class implements the same algorithm (and more):

In [ ]:
from sklearn.mixture import GaussianMixture

gm = GaussianMixture(n_components=2, random_state=42, tol=1e-8)
gm.fit(y.reshape(-1, 1))

print("scikit-learn estimates:")
print(f"  pi:     {gm.weights_[0]:.3f}")
print(f"  mu1:    {gm.means_[0, 0]:.2f}")
print(f"  sigma1: {np.sqrt(gm.covariances_[0, 0, 0]):.2f}")
print(f"  mu2:    {gm.means_[1, 0]:.2f}")
print(f"  sigma2: {np.sqrt(gm.covariances_[1, 0, 0]):.2f}")
print(f"\nOur EM estimates:")
print(f"  pi:     {result['pi']:.3f}")
print(f"  mu1:    {result['mu1']:.2f}")
print(f"  sigma1: {result['sigma1']:.2f}")
print(f"  mu2:    {result['mu2']:.2f}")
print(f"  sigma2: {result['sigma2']:.2f}")

The estimates may differ slightly due to initialization or label ordering (component 1 in our implementation might correspond to component 2 in scikit-learn), but the final log-likelihood values should be very close.

**Extension to K components:** The derivation generalizes directly to $K > 2$ components. Each observation has $K$ responsibilities that sum to 1, and the M-step computes $K$ sets of weighted MLEs. The main practical difficulty is choosing $K$ (model selection) and the increased risk of local optima.

### Question

Suppose we have $n = 4$ observations from a two-component Gaussian mixture: $y = (60, 65, 68, 73)$. The current parameter estimates are $\pi^{(t)} = 0.5$, $\mu_1^{(t)} = 61$, $\sigma_1^{(t)} = 2$, $\mu_2^{(t)} = 70$, $\sigma_2^{(t)} = 3$.

(a) Compute the responsibility $\gamma_3$ for observation $y_3 = 68$.

(b) If we increased $\sigma_1^{(t)}$ from 2 to 5 (keeping everything else fixed), would $\gamma_3$ increase or decrease? Explain intuitively.

(c) If we increased $\pi^{(t)}$ from 0.5 to 0.8, would $\gamma_3$ increase or decrease?

### Answer

(a) We compute:
- $\phi(68 | 61, 2) = \frac{1}{\sqrt{2\pi} \cdot 2} \exp\left(-\frac{(68-61)^2}{2 \cdot 4}\right) = \frac{1}{2\sqrt{2\pi}} e^{-49/8} \approx 0.00443 \cdot e^{-6.125} \approx 0.00097$

Wait, let us be more precise. $\phi(68 | 61, 2) = \text{dnorm}(68, 61, 2)$. The z-score is $(68 - 61)/2 = 3.5$, so $\phi(68 | 61, 2) \approx 0.001260$.

$\phi(68 | 70, 3)$: z-score is $(68 - 70)/3 \approx -0.667$, so $\phi(68 | 70, 3) \approx 0.1065$.

$\gamma_3 = \frac{0.5 \times 0.001260}{0.5 \times 0.001260 + 0.5 \times 0.1065} = \frac{0.000630}{0.000630 + 0.05325} \approx 0.0117$

So $\gamma_3 \approx 0.012$: the observation 68 is overwhelmingly assigned to component 2.

(b) Increasing $\sigma_1$ from 2 to 5 makes the first component wider. The density $\phi(68 | 61, 5)$ would be much larger than $\phi(68 | 61, 2)$ since 68 is no longer 3.5 standard deviations away from $\mu_1$, but only 1.4 standard deviations. So $\gamma_3$ would **increase** (observation 68 becomes more plausible under the wider component 1).

(c) Increasing $\pi$ from 0.5 to 0.8 puts more prior weight on component 1. The numerator $\pi \cdot \phi(68 | 61, 2)$ increases by a factor of 0.8/0.5 = 1.6, while the denominator's second term $(1-\pi) \cdot \phi(68 | 70, 3)$ decreases. So $\gamma_3$ would **increase**, though only slightly here because the likelihood ratio is so extreme that the prior weight barely matters.

## Convergence Properties

### Monotone Likelihood Increase

As stated earlier, the EM algorithm guarantees:

$$\ell(\boldsymbol{\theta}^{(t+1)}) \geq \ell(\boldsymbol{\theta}^{(t)})$$

To see why, write the observed-data log-likelihood using the complete-data distribution:

$$\ell(\boldsymbol{\theta}) = \log f(\mathbf{Y} | \boldsymbol{\theta}) = \log \frac{f(\mathbf{Y}, \mathbf{Z} | \boldsymbol{\theta})}{f(\mathbf{Z} | \mathbf{Y}, \boldsymbol{\theta})}$$

Taking the expectation of both sides with respect to $f(\mathbf{Z} | \mathbf{Y}, \boldsymbol{\theta}^{(t)})$:

$$\ell(\boldsymbol{\theta}) = Q(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)}) - H(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)})$$

The M-step ensures $Q(\boldsymbol{\theta}^{(t+1)} | \boldsymbol{\theta}^{(t)}) \geq Q(\boldsymbol{\theta}^{(t)} | \boldsymbol{\theta}^{(t)})$. By the information inequality (Gibbs' inequality), $H(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)}) \leq H(\boldsymbol{\theta}^{(t)} | \boldsymbol{\theta}^{(t)})$ for all $\boldsymbol{\theta}$, which means $-H(\boldsymbol{\theta}^{(t+1)} | \boldsymbol{\theta}^{(t)}) \geq -H(\boldsymbol{\theta}^{(t)} | \boldsymbol{\theta}^{(t)})$. Adding both inequalities yields $\ell(\boldsymbol{\theta}^{(t+1)}) \geq \ell(\boldsymbol{\theta}^{(t)})$.

This monotone increase means the sequence of log-likelihood values is non-decreasing. If the log-likelihood is bounded above (as it typically is), the sequence must converge. However, convergence of the log-likelihood does *not* guarantee convergence to the global maximum.

### Convergence Rate

EM converges **linearly**, meaning that the error decreases by a constant fraction at each iteration:

$$\|\boldsymbol{\theta}^{(t+1)} - \hat{\boldsymbol{\theta}}\| \approx r \, \|\boldsymbol{\theta}^{(t)} - \hat{\boldsymbol{\theta}}\|$$

where $r \in (0, 1)$ is the rate of convergence. Compare this to Newton-Raphson, which converges **quadratically** (the error is squared at each step).

The rate $r$ depends on the "fraction of missing information." Intuitively, the more information the latent variables carry relative to the observed data, the more the E-step is "guessing" and the slower convergence will be. When the components overlap heavily, the latent assignments are more uncertain, and EM converges more slowly.

### Stopping Criteria

Common stopping criteria include:

1. **Log-likelihood change:** $|\ell(\boldsymbol{\theta}^{(t+1)}) - \ell(\boldsymbol{\theta}^{(t)})| < \epsilon$
2. **Relative change:** $|\ell(\boldsymbol{\theta}^{(t+1)}) - \ell(\boldsymbol{\theta}^{(t)})| / |\ell(\boldsymbol{\theta}^{(t)})| < \epsilon$
3. **Parameter change:** $\|\boldsymbol{\theta}^{(t+1)} - \boldsymbol{\theta}^{(t)}\| < \epsilon$
4. **Maximum iterations:** stop after a fixed number of iterations

In practice, combining a likelihood-based criterion with a maximum iteration limit is a good default.

### Local Maxima and Initialization

The mixture log-likelihood is not globally concave and may have multiple local maxima. EM is guaranteed to converge to a *stationary point*, but this could be a local rather than global maximum. Initialization strategies to mitigate this include:

- **Random restarts:** Run EM from many different initial parameter values and keep the solution with the highest final log-likelihood.
- **K-means initialization:** Use k-means clustering to get initial component assignments, then compute initial parameter estimates from these clusters.
- **Hierarchical initialization:** Start with a single component and split iteratively.

### Convergence Speed: Well-Separated vs. Overlapping Components

The following code compares EM convergence for two scenarios: well-separated components (easy) and highly overlapping components (hard).

In [ ]:
np.random.seed(42)

scenarios = {
    "Well-separated ($\\mu_1=55, \\mu_2=75$)": {
        "mu1": 55, "mu2": 75, "sigma1": 3, "sigma2": 3
    },
    "Overlapping ($\\mu_1=63, \\mu_2=67$)": {
        "mu1": 63, "mu2": 67, "sigma1": 3, "sigma2": 3
    },
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for idx, (label, params) in enumerate(scenarios.items()):
    z = np.random.binomial(1, 0.4, 300)
    y_sim = np.where(
        z == 0,
        np.random.normal(params["mu1"], params["sigma1"], 300),
        np.random.normal(params["mu2"], params["sigma2"], 300),
    )
    res = em_gmm(
        y_sim, pi_init=0.5,
        mu1_init=params["mu1"] - 5, sigma1_init=5,
        mu2_init=params["mu2"] + 5, sigma2_init=5,
        max_iter=200
    )
    axes[idx].plot(res["loglik_history"], "o-", markersize=3, color="steelblue")
    axes[idx].set_xlabel("Iteration")
    axes[idx].set_ylabel("Log-likelihood")
    axes[idx].set_title(label)

plt.tight_layout()


### Multiple Initializations

Since EM can converge to different local maxima depending on where it starts, a standard practice is to run the algorithm from many random initializations and keep the solution with the highest final log-likelihood. The code below runs EM 50 times with random starting values on the overlapping-component data and records the final log-likelihood from each run. The histogram reveals how many distinct modes the algorithm finds and how much the final log-likelihood varies across initializations.

In [ ]:
np.random.seed(42)

# Use the overlapping scenario
z_overlap = np.random.binomial(1, 0.4, 300)
y_overlap = np.where(
    z_overlap == 0,
    np.random.normal(63, 3, 300),
    np.random.normal(67, 3, 300),
)

final_logliks = []
np.random.seed(0)
for _ in range(50):
    mu1_init = np.random.uniform(55, 75)
    mu2_init = np.random.uniform(55, 75)
    s_init = np.random.uniform(1, 6)
    res = em_gmm(
        y_overlap, pi_init=0.5,
        mu1_init=mu1_init, sigma1_init=s_init,
        mu2_init=mu2_init, sigma2_init=s_init,
        max_iter=500
    )
    final_logliks.append(res["loglik_history"][-1])

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(final_logliks, bins=20, edgecolor="white", color="steelblue")
ax.set_xlabel("Final Log-Likelihood")
ax.set_ylabel("Count")
ax.set_title("Final Log-Likelihoods from 50 Random Initializations (Overlapping Components)")
plt.tight_layout()


print(f"Best log-likelihood: {max(final_logliks):.2f}")
print(f"Worst log-likelihood: {min(final_logliks):.2f}")

## EM for Missing Data

The EM algorithm is not limited to mixture models. It applies broadly to any setting with missing or incomplete data. We now consider a classical application: estimating the mean and covariance of a bivariate normal distribution when one variable has missing values.

### Setup

Suppose $(X_i, Y_i) \sim N(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ where

$$\boldsymbol{\mu} = \begin{pmatrix} \mu_X \\ \mu_Y \end{pmatrix}, \quad \boldsymbol{\Sigma} = \begin{pmatrix} \sigma_X^2 & \rho \sigma_X \sigma_Y \\ \rho \sigma_X \sigma_Y & \sigma_Y^2 \end{pmatrix}$$

For some observations, $Y_i$ is missing at random (MAR). We observe $X_i$ for all subjects but $Y_i$ for only a subset.

### E-step: Conditional Expectations

For observations where $Y_i$ is observed, the sufficient statistics $Y_i$ and $Y_i^2$ are known. For observations where $Y_i$ is missing, the E-step replaces them with their conditional expectations given $X_i$ and the current parameter estimates.

From the properties of the bivariate normal:

$$E[Y_i | X_i, \boldsymbol{\theta}^{(t)}] = \mu_Y^{(t)} + \rho^{(t)} \frac{\sigma_Y^{(t)}}{\sigma_X^{(t)}} (X_i - \mu_X^{(t)})$$

$$E[Y_i^2 | X_i, \boldsymbol{\theta}^{(t)}] = \text{Var}(Y_i | X_i, \boldsymbol{\theta}^{(t)}) + \left(E[Y_i | X_i, \boldsymbol{\theta}^{(t)}]\right)^2$$

where $\text{Var}(Y_i | X_i, \boldsymbol{\theta}^{(t)}) = \sigma_Y^{2(t)} (1 - \rho^{2(t)})$.

### M-step: Update Parameters

With the sufficient statistics filled in for missing observations, the M-step computes the standard normal MLEs:

$$\mu_X^{(t+1)} = \frac{1}{n} \sum_{i=1}^n X_i, \quad \mu_Y^{(t+1)} = \frac{1}{n} \sum_{i=1}^n \tilde{Y}_i$$

$$\sigma_X^{2(t+1)} = \frac{1}{n} \sum_{i=1}^n (X_i - \mu_X^{(t+1)})^2, \quad \sigma_Y^{2(t+1)} = \frac{1}{n} \sum_{i=1}^n \widetilde{Y_i^2} - (\mu_Y^{(t+1)})^2$$

where $\tilde{Y}_i = Y_i$ if observed and $\tilde{Y}_i = E[Y_i | X_i, \boldsymbol{\theta}^{(t)}]$ if missing, and similarly $\widetilde{Y_i^2} = Y_i^2$ if observed and $\widetilde{Y_i^2} = E[Y_i^2 | X_i, \boldsymbol{\theta}^{(t)}]$ if missing.

The cross-product term for estimating $\rho$ uses $\widetilde{X_i Y_i} = X_i Y_i$ if observed and $X_i \cdot E[Y_i | X_i, \boldsymbol{\theta}^{(t)}]$ if missing.

### Implementation

In [ ]:
def em_bivariate_normal(X, Y, observed, max_iter=200, tol=1e-8):
    """EM for bivariate normal with missing Y values.

    Parameters
    ----------
    X : array of shape (n,)
        Fully observed variable.
    Y : array of shape (n,)
        Partially observed variable (NaN for missing values).
    observed : boolean array of shape (n,)
        True if Y_i is observed.
    max_iter : int
        Maximum number of iterations.
    tol : float
        Convergence tolerance.

    Returns
    -------
    dict with estimated parameters and convergence history.
    """
    n = len(X)
    obs = observed
    mis = ~observed

    # Initialize from complete cases
    mu_x = np.mean(X)
    mu_y = np.mean(Y[obs])
    var_x = np.var(X)
    var_y = np.var(Y[obs])
    cov_xy = np.cov(X[obs], Y[obs])[0, 1]
    rho = cov_xy / (np.sqrt(var_x) * np.sqrt(var_y))

    history = []

    for iteration in range(max_iter):
        sig_x = np.sqrt(var_x)
        sig_y = np.sqrt(var_y)

        # E-step: fill in missing Y values
        Y_hat = Y.copy()
        Y2_hat = Y.copy() ** 2
        XY_hat = X * Y.copy()

        # Conditional expectations for missing observations
        cond_mean = mu_y + rho * (sig_y / sig_x) * (X[mis] - mu_x)
        cond_var = var_y * (1 - rho ** 2)

        Y_hat[mis] = cond_mean
        Y2_hat[mis] = cond_var + cond_mean ** 2
        XY_hat[mis] = X[mis] * cond_mean

        # M-step: update parameters
        mu_x_new = np.mean(X)
        mu_y_new = np.mean(Y_hat)
        var_x_new = np.mean((X - mu_x_new) ** 2)
        var_y_new = np.mean(Y2_hat) - mu_y_new ** 2
        cov_xy_new = np.mean(XY_hat) - mu_x_new * mu_y_new
        rho_new = cov_xy_new / (np.sqrt(var_x_new) * np.sqrt(var_y_new))

        # Store parameter history
        history.append({
            "mu_x": mu_x_new, "mu_y": mu_y_new,
            "var_x": var_x_new, "var_y": var_y_new,
            "rho": rho_new,
        })

        # Check convergence
        if iteration > 0:
            param_change = max(
                abs(mu_x_new - mu_x), abs(mu_y_new - mu_y),
                abs(var_x_new - var_x), abs(var_y_new - var_y),
                abs(rho_new - rho),
            )
            if param_change < tol:
                break

        mu_x, mu_y = mu_x_new, mu_y_new
        var_x, var_y = var_x_new, var_y_new
        rho = rho_new

    return {
        "mu_x": mu_x, "mu_y": mu_y,
        "var_x": var_x, "var_y": var_y,
        "rho": rho,
        "history": history,
    }

### Simulation Study

In [ ]:
np.random.seed(42)

# True parameters
n = 500
mu_true = np.array([5.0, 10.0])
rho_true = 0.7
sigma_x_true, sigma_y_true = 2.0, 3.0
Sigma_true = np.array([
    [sigma_x_true ** 2, rho_true * sigma_x_true * sigma_y_true],
    [rho_true * sigma_x_true * sigma_y_true, sigma_y_true ** 2],
])

# Generate complete data
data = np.random.multivariate_normal(mu_true, Sigma_true, n)
X_full, Y_full = data[:, 0], data[:, 1]

# Introduce 30% MAR missingness in Y (missingness depends on X)
prob_missing = 1 / (1 + np.exp(-0.5 * (X_full - np.median(X_full))))
prob_missing = prob_missing / prob_missing.max() * 0.45  # scale to ~30%
missing = np.random.binomial(1, prob_missing, n).astype(bool)
Y_obs = Y_full.copy()
Y_obs[missing] = np.nan
observed = ~missing

print(f"Number of observations: {n}")
print(f"Number missing: {missing.sum()} ({missing.mean()*100:.0f}%)")

# Run EM
em_result = em_bivariate_normal(X_full, Y_obs, observed)

# Complete-case analysis
cc_mu_x = np.mean(X_full[observed])
cc_mu_y = np.mean(Y_full[observed])
cc_var_x = np.var(X_full[observed])
cc_var_y = np.var(Y_full[observed])
cc_rho = np.corrcoef(X_full[observed], Y_full[observed])[0, 1]

# Full-data analysis (oracle)
full_mu_x = np.mean(X_full)
full_mu_y = np.mean(Y_full)
full_var_x = np.var(X_full)
full_var_y = np.var(Y_full)
full_rho = np.corrcoef(X_full, Y_full)[0, 1]

# Compare
print(f"\n{'Parameter':<10s} {'True':>8s} {'Full data':>10s} {'EM':>8s} {'Complete-case':>14s}")
print("-" * 55)
for name, true_val, full_val, em_val, cc_val in [
    ("mu_x", mu_true[0], full_mu_x, em_result["mu_x"], cc_mu_x),
    ("mu_y", mu_true[1], full_mu_y, em_result["mu_y"], cc_mu_y),
    ("var_x", sigma_x_true**2, full_var_x, em_result["var_x"], cc_var_x),
    ("var_y", sigma_y_true**2, full_var_y, em_result["var_y"], cc_var_y),
    ("rho", rho_true, full_rho, em_result["rho"], cc_rho),
]:
    print(f"{name:<10s} {true_val:>8.3f} {full_val:>10.3f} {em_val:>8.3f} {cc_val:>14.3f}")

print(f"\nEM iterations: {len(em_result['history'])}")

### Discussion: EM vs. Simple Alternatives

**Complete-case analysis** discards all observations with any missing values. This wastes data and can introduce bias when data are not missing completely at random (MCAR). In our MAR simulation, complete-case estimates of $\mu_Y$ may be biased because the missingness depends on $X$.

**Mean imputation** replaces missing $Y_i$ with $\bar{Y}_{\text{observed}}$, ignoring the relationship between $X$ and $Y$. This artificially reduces variance and attenuates correlations.

**EM** uses the conditional distribution $Y | X$ to impute the missing values (and their second moments), preserving the correlation structure. It produces consistent estimates under the MAR assumption without wasting any observed data.

In [ ]:
# Mean imputation for comparison
Y_mean_imp = Y_obs.copy()
Y_mean_imp[missing] = np.nanmean(Y_obs)
mi_rho = np.corrcoef(X_full, Y_mean_imp)[0, 1]

print(f"Correlation estimates:")
print(f"  True:           {rho_true:.3f}")
print(f"  Full data:      {full_rho:.3f}")
print(f"  EM:             {em_result['rho']:.3f}")
print(f"  Complete-case:  {cc_rho:.3f}")
print(f"  Mean imputation: {mi_rho:.3f}")

Mean imputation attenuates the correlation because it replaces missing $Y$ values with a constant, destroying the $X$-$Y$ relationship for those observations.

### Question

Suppose you have a trivariate normal $(X_1, X_2, X_3)$ where $X_3$ has 25% missing values but $X_1$ and $X_2$ are fully observed.

(a) In the E-step of EM, what information does the algorithm use to fill in missing $X_3$ values? Write the form of the conditional expectation.

(b) A colleague suggests imputing missing $X_3$ values with the marginal mean $\bar{X}_3$ (computed from observed $X_3$ values) and then estimating the full covariance matrix. Why is this inferior to EM?

### Answer

(a) The E-step computes $E[X_3 | X_1, X_2, \boldsymbol{\theta}^{(t)}]$, which uses the multivariate conditional normal formula:

$$E[X_3 | X_1, X_2] = \mu_3 + \boldsymbol{\Sigma}_{3,(1,2)} \boldsymbol{\Sigma}_{(1,2),(1,2)}^{-1} \begin{pmatrix} X_1 - \mu_1 \\ X_2 - \mu_2 \end{pmatrix}$$

This is a linear function of both $X_1$ and $X_2$. The algorithm uses the correlation structure between $X_3$ and the other two variables to make informed predictions about the missing values.

(b) Marginal mean imputation replaces all missing $X_3$ values with a single constant $\bar{X}_3$. This has two problems. First, it ignores the information in $(X_1, X_2)$ that is predictive of $X_3$. An observation with extreme $X_1$ and $X_2$ values will get the same imputed $X_3$ as one with average values, even though the conditional expectation differs. Second, it artificially reduces $\text{Var}(X_3)$ and attenuates all correlations involving $X_3$, because it replaces natural variability with a constant. EM avoids both issues: it uses the conditional distribution to impute values (preserving the mean structure) and it accounts for the conditional variance $\text{Var}(X_3 | X_1, X_2) = \sigma_3^2(1 - R_{3 \cdot 12}^2)$ in the sufficient statistics (preserving the variance and correlation structure).

## Monte Carlo EM

In all examples so far, the E-step had a closed-form expression: responsibilities in the GMM, conditional moments in the bivariate normal. In more complex models, such as generalized linear mixed models (GLMMs), spatial models, or Bayesian models with non-conjugate priors, the conditional expectation $E[\log f(\mathbf{Y}, \mathbf{Z} | \boldsymbol{\theta}) | \mathbf{Y}, \boldsymbol{\theta}^{(t)}]$ may involve intractable integrals with no analytical solution. Monte Carlo EM (MCEM), introduced by Wei and Tanner (1990), addresses this by replacing the exact E-step with a Monte Carlo approximation.

### Formulation

Recall that the Q-function is an integral over the latent variable space:

$$Q(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)}) = \int \log f(\mathbf{Y}, \mathbf{Z} | \boldsymbol{\theta}) \, f(\mathbf{Z} | \mathbf{Y}, \boldsymbol{\theta}^{(t)}) \, d\mathbf{Z}$$

When this integral is intractable, we approximate it by drawing $M$ samples $\mathbf{Z}^{(1)}, \ldots, \mathbf{Z}^{(M)}$ from the conditional distribution $f(\mathbf{Z} | \mathbf{Y}, \boldsymbol{\theta}^{(t)})$ and computing:

$$\hat{Q}(\boldsymbol{\theta} | \boldsymbol{\theta}^{(t)}) = \frac{1}{M} \sum_{m=1}^{M} \log f(\mathbf{Y}, \mathbf{Z}^{(m)} | \boldsymbol{\theta})$$

The M-step then maximizes $\hat{Q}$ instead of $Q$. As $M \to \infty$, $\hat{Q} \to Q$ by the law of large numbers, so the MCEM iterates approach exact EM iterates.

### Pseudocode

```
MCEM Algorithm:
1. Initialize parameters θ^(0), set Monte Carlo sample size M
2. For t = 0, 1, 2, ... until convergence:
   a. MC E-step: Draw Z^(1), ..., Z^(M) ~ f(Z | Y, θ^(t))
      Compute Q_hat(θ | θ^(t)) = (1/M) Σ log f(Y, Z^(m) | θ)
   b. M-step: Set θ^(t+1) = argmax_θ Q_hat(θ | θ^(t))
3. Return θ^(t+1)
```

### Example: MCEM for Gaussian Mixture Model

To illustrate, we apply MCEM to the same two-component GMM from the Motivation section. Instead of computing the analytic responsibilities $\gamma_i$, we sample latent indicators $z_i \sim \text{Bernoulli}(\gamma_i)$ and use those hard assignments in the M-step. This is a toy example where exact EM is available, allowing us to compare the two approaches directly.

In [ ]:
def mcem_gmm(y, pi_init, mu1_init, sigma1_init, mu2_init, sigma2_init,
             M=100, max_iter=200, tol=1e-6):
    """MCEM algorithm for a 2-component Gaussian mixture model.

    Instead of computing exact responsibilities, we draw M samples of latent
    indicators and average the complete-data sufficient statistics.

    Parameters
    ----------
    y : array of shape (n,)
    M : int
        Number of Monte Carlo samples per E-step.
    """
    n = len(y)
    pi = pi_init
    mu1, mu2 = mu1_init, mu2_init
    s1, s2 = sigma1_init, sigma2_init
    loglik_history = []

    for iteration in range(max_iter):
        # Compute exact responsibilities (used for sampling)
        d1 = pi * stats.norm.pdf(y, mu1, s1)
        d2 = (1 - pi) * stats.norm.pdf(y, mu2, s2)
        gamma = d1 / (d1 + d2)

        # Observed-data log-likelihood (for monitoring)
        ll = np.sum(np.log(d1 + d2))
        loglik_history.append(ll)

        if iteration > 0 and abs(loglik_history[-1] - loglik_history[-2]) < tol:
            break

        # MC E-step: draw M sets of latent indicators
        # z_im = 1 means observation i assigned to component 1 in sample m
        z_samples = np.random.binomial(1, gamma[:, None], size=(n, M))

        # Average sufficient statistics over M samples
        n1 = z_samples.sum(axis=0).mean()
        n2 = n - n1

        pi = n1 / n
        mu1 = (z_samples * y[:, None]).sum(axis=0).mean() / n1
        mu2 = ((1 - z_samples) * y[:, None]).sum(axis=0).mean() / n2
        s1 = np.sqrt((z_samples * (y[:, None] - mu1) ** 2).sum(axis=0).mean() / n1)
        s2 = np.sqrt(((1 - z_samples) * (y[:, None] - mu2) ** 2).sum(axis=0).mean() / n2)

    return {"pi": pi, "mu1": mu1, "sigma1": s1, "mu2": mu2, "sigma2": s2,
            "loglik_history": loglik_history}

We now compare exact EM and MCEM with different values of $M$:

In [ ]:
np.random.seed(42)

# Run exact EM
result_exact = em_gmm(y, pi_init=0.5, mu1_init=58, sigma1_init=4,
                       mu2_init=72, sigma2_init=4, max_iter=100)

# Run MCEM with different M values
M_values = [1, 10, 100]
mcem_results = {}
for M in M_values:
    np.random.seed(42)
    mcem_results[M] = mcem_gmm(y, pi_init=0.5, mu1_init=58, sigma1_init=4,
                                mu2_init=72, sigma2_init=4, M=M, max_iter=100)

# Plot log-likelihood trajectories
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(result_exact["loglik_history"], "k-", linewidth=2, label="Exact EM")
colors = ["tab:orange", "tab:blue", "tab:green"]
for color, M in zip(colors, M_values):
    ll = mcem_results[M]["loglik_history"]
    ax.plot(ll, "--", color=color, alpha=0.8, label=f"MCEM (M={M})")
ax.set_xlabel("Iteration")
ax.set_ylabel("Log-likelihood")
ax.set_title("Exact EM vs. MCEM: Log-Likelihood Trajectories")
ax.legend()
plt.xlim(0, 20)
plt.tight_layout()


### Practical Considerations

- **MC noise vs. cost tradeoff:** Small $M$ is cheap per iteration but introduces variance that can prevent convergence. Large $M$ is accurate but expensive. A common strategy is to start with small $M$ (for fast early progress) and increase $M$ as the algorithm nears convergence.
- **Convergence detection:** Because of Monte Carlo noise, the log-likelihood is no longer guaranteed to increase monotonically. Convergence criteria must account for this variability, for example by checking whether the change is small relative to the MC standard error, or by requiring the criterion to be met several consecutive times.
- **MCMC for complex posteriors:** When $f(\mathbf{Z} | \mathbf{Y}, \boldsymbol{\theta}^{(t)})$ itself cannot be sampled from directly, Markov chain Monte Carlo (MCMC) methods (e.g., Gibbs sampling, Metropolis-Hastings) can be used to draw approximate samples. This introduces additional approximation and correlation between samples.

## Applied Example: Zero-Inflated Poisson

Count data often exhibit more zeros than a standard Poisson model can accommodate. The **zero-inflated Poisson (ZIP)** model (Lambert, 1992) handles this by assuming each observation comes from one of two latent states: a "structural zero" state (with probability $\pi$) that always produces $y_i = 0$, and a Poisson state (with probability $1 - \pi$) that generates counts from $\text{Poisson}(\lambda)$. The PMF is:

$$P(Y_i = y) = \begin{cases} \pi + (1 - \pi) e^{-\lambda}, & y = 0 \\ (1 - \pi) \frac{e^{-\lambda} \lambda^y}{y!}, & y > 0 \end{cases}$$

The parameters $\pi$ and $\lambda$ can be estimated via EM by treating the structural-zero indicator as a latent variable.

### EM Derivation

Let $z_i = 1$ if observation $i$ is a structural zero and $z_i = 0$ if it comes from the Poisson component. The complete-data log-likelihood is:

$$\ell_c(\pi, \lambda) = \sum_{i=1}^n \left[ z_i \log \pi + (1 - z_i) \left( \log(1 - \pi) + y_i \log \lambda - \lambda - \log(y_i!) \right) \right]$$

**E-step:** Compute the posterior probability that observation $i$ is a structural zero:

$$\gamma_i^{(t)} = E[z_i | y_i, \pi^{(t)}, \lambda^{(t)}]$$

For $y_i > 0$, the observation cannot be a structural zero, so $\gamma_i = 0$. For $y_i = 0$:

$$\gamma_i^{(t)} = \frac{\pi^{(t)}}{\pi^{(t)} + (1 - \pi^{(t)}) e^{-\lambda^{(t)}}}$$

**M-step:** Update parameters using the expected sufficient statistics:

$$\pi^{(t+1)} = \frac{1}{n} \sum_{i=1}^n \gamma_i^{(t)}, \quad \lambda^{(t+1)} = \frac{\sum_{i=1}^n (1 - \gamma_i^{(t)}) y_i}{\sum_{i=1}^n (1 - \gamma_i^{(t)})}$$

The update for $\lambda$ is a weighted mean of the $y_i$ values, weighted by the probability of belonging to the Poisson component.

### Implementation

In [ ]:
def em_zip(y, pi_init=0.3, lam_init=1.0, max_iter=200, tol=1e-8):
    """EM algorithm for the zero-inflated Poisson model.

    Parameters
    ----------
    y : array of shape (n,)
        Observed count data.
    pi_init : float
        Initial structural-zero probability.
    lam_init : float
        Initial Poisson rate.

    Returns
    -------
    dict with keys: pi, lam, loglik_history
    """
    n = len(y)
    pi = pi_init
    lam = lam_init
    loglik_history = []
    is_zero = (y == 0)

    for iteration in range(max_iter):
        # E-step: compute gamma_i
        gamma = np.zeros(n)
        gamma[is_zero] = pi / (pi + (1 - pi) * np.exp(-lam))
        # gamma[~is_zero] = 0 by initialization

        # Observed-data log-likelihood
        ll_zero = np.log(pi + (1 - pi) * np.exp(-lam)) * is_zero.sum()
        ll_pos = np.sum(~is_zero * (
            np.log(1 - pi) + y * np.log(lam) - lam
            - np.array([np.sum(np.log(np.arange(1, yi + 1))) for yi in y])
        ))
        ll = ll_zero + ll_pos
        loglik_history.append(ll)

        if iteration > 0 and abs(loglik_history[-1] - loglik_history[-2]) < tol:
            break

        # M-step
        pi = np.mean(gamma)
        lam = np.sum((1 - gamma) * y) / np.sum(1 - gamma)

    return {"pi": pi, "lam": lam, "loglik_history": loglik_history}


# Simulate ZIP data
np.random.seed(42)
n_zip = 1000
pi_true_zip = 0.3
lam_true_zip = 2.5

z_zip = np.random.binomial(1, pi_true_zip, n_zip)  # 1 = structural zero
y_zip = np.where(z_zip == 1, 0, np.random.poisson(lam_true_zip, n_zip))

print(f"Simulated data: n={n_zip}, observed zeros={np.sum(y_zip == 0)}, "
      f"expected Poisson zeros={int(n_zip * (1 - pi_true_zip) * np.exp(-lam_true_zip))}")

# Run EM
zip_result = em_zip(y_zip, pi_init=0.5, lam_init=1.0)
print(f"\nEM estimates vs. true values:")
print(f"  pi:     {zip_result['pi']:.3f}  (true: {pi_true_zip})")
print(f"  lambda: {zip_result['lam']:.3f}  (true: {lam_true_zip})")
print(f"  Iterations: {len(zip_result['loglik_history'])}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(zip_result["loglik_history"], "o-", markersize=3, color="steelblue")
ax.set_xlabel("Iteration")
ax.set_ylabel("Log-likelihood")
ax.set_title("EM Convergence for Zero-Inflated Poisson")
plt.tight_layout()


### Question

How would the E-step and M-step change if we extended the ZIP model to a ZIP regression, where $\lambda_i = \exp(\mathbf{X}_i \boldsymbol{\beta})$ depends on covariates?

### Answer

The E-step would become observation-specific in $\lambda$: for $y_i = 0$, the responsibility would be $\gamma_i = \pi / [\pi + (1 - \pi) e^{-\lambda_i^{(t)}}]$ where $\lambda_i^{(t)} = \exp(\mathbf{X}_i \boldsymbol{\beta}^{(t)})$. Each zero gets a different posterior probability of being structural, depending on its covariate values.

The M-step for $\pi$ stays the same: $\pi^{(t+1)} = \bar{\gamma}$. However, the M-step for $\boldsymbol{\beta}$ no longer has a closed form. Instead, it requires maximizing a weighted Poisson log-likelihood: $\sum_{i=1}^n (1 - \gamma_i) [y_i \mathbf{X}_i \boldsymbol{\beta} - \exp(\mathbf{X}_i \boldsymbol{\beta})]$ with respect to $\boldsymbol{\beta}$. This is equivalent to fitting a weighted Poisson GLM with prior weights $(1 - \gamma_i)$, which can be done using standard GLM software (e.g., IRLS). The modularity of EM makes this straightforward: the M-step reduces to a familiar weighted regression problem.

## Summary

The EM algorithm is a general strategy for maximum likelihood estimation with incomplete or latent data. We applied it to Gaussian mixture models, bivariate normal with missing data, zero-inflated Poisson models, and introduced Monte Carlo EM for settings where the E-step is intractable.

**Pros:**

- **Numerically stable:** For exact EM, the monotone likelihood increase property guarantees that each iteration improves (or maintains) the observed-data log-likelihood, unlike gradient-based methods that can overshoot. (Approximate variants like MCEM sacrifice this guarantee in exchange for tractability.)
- **Easy to implement and modular:** The E-step and M-step use standard tools (Bayes' rule, weighted MLEs, off-the-shelf GLM software). New models can be accommodated by swapping in a different M-step.
- **No gradients or Hessians required:** EM avoids the need to derive or compute first and second derivatives of the observed-data log-likelihood, which can be complex for latent-variable models.
- **Memory efficient:** Unlike Newton-Raphson, EM does not need to store or invert an information matrix.
- **Parameter constraints handled naturally:** Update formulas for mixing proportions, variances, and probabilities automatically satisfy their constraints (e.g., $\pi \in [0, 1]$, $\sigma^2 > 0$) without requiring reparameterization or constrained optimization.

**Cons:**

- **Linear convergence:** EM converges linearly, which can be slow when components overlap heavily or the fraction of missing information is large. Newton-Raphson achieves quadratic convergence.
- **No direct standard errors:** The EM algorithm does not produce a Hessian or information matrix as a byproduct. Obtaining standard errors requires additional work, such as Louis' method, the supplemented EM algorithm, or bootstrap resampling.
- **No guarantee of global maximum:** Like other iterative methods, EM can converge to local maxima or saddle points. Multiple random initializations are essential.
- **E-step may be intractable:** For complex models (GLMMs, non-conjugate Bayesian models), the conditional expectation in the E-step may have no closed form, requiring approximations such as MCEM or variational methods.

### References

- Dempster, A. P., Laird, N. M., & Rubin, D. B. (1977). Maximum likelihood from incomplete data via the EM algorithm. *Journal of the Royal Statistical Society: Series B*, 39(1), 1-38.
- Wu, C. F. J. (1983). On the convergence properties of the EM algorithm. *The Annals of Statistics*, 11(1), 95-103.
- McLachlan, G. J., & Krishnan, T. (2008). *The EM Algorithm and Extensions* (2nd ed.). Wiley.
- Wei, G. C. G., & Tanner, M. A. (1990). A Monte Carlo implementation of the EM algorithm and the poor man's data augmentation algorithms. *Journal of the American Statistical Association*, 85(411), 699-704.
- Lambert, D. (1992). Zero-inflated Poisson regression, with an application to defects in manufacturing. *Technometrics*, 34(1), 1-14.